# SCRFD Kaggle Packed Env Runner

Attach three Kaggle Datasets before running:
- source repo dataset containing `scrfd-fullsearch-kaggle-offline.zip`
- WIDER FACE dataset containing `retinaface/`
- packed environment dataset containing `scrfd-kaggle-baseline-py312-cu128.tar.gz`


In [ ]:
REPO_SOURCE = ""
DATA_ROOT = ""
PACKED_ENV_ARCHIVE = ""

RUN_MODE = "baseline_train_quick"
GPU_ID = 0
TOTAL_EPOCHS = 16
EVAL_INTERVAL = 4
CHECKPOINT_INTERVAL = 4
IDX_FROM = 1
IDX_TO = 64
TEMPLATE = 10
CHECKPOINT = ""
THR = "0.02"
MODE_VALUE = "0"


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

def find_repo_source(explicit_path: str) -> Path:
    if explicit_path:
        p = Path(explicit_path)
        if p.exists():
            return p
    input_root = Path('/kaggle/input')
    for pattern in ['**/scrfd-fullsearch-kaggle-offline.zip', '**/scrfd-fullsearch-kaggle-kagglefix-v2.zip']:
        matches = list(input_root.glob(pattern))
        if matches:
            return matches[0]
    for candidate in input_root.glob('**/scrfd-fullsearch-kaggle'):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('Could not find SCRFD source dataset in /kaggle/input')

def find_data_root(explicit_path: str) -> Path:
    if explicit_path:
        p = Path(explicit_path)
        if p.exists():
            return p
    input_root = Path('/kaggle/input')
    for candidate in input_root.glob('**/retinaface'):
        if (candidate / 'train' / 'labelv2.txt').exists() and (candidate / 'val' / 'labelv2.txt').exists():
            return candidate
    raise FileNotFoundError('Could not find retinaface dataset root in /kaggle/input')

def find_packed_env(explicit_path: str) -> Path:
    if explicit_path:
        p = Path(explicit_path)
        if p.exists():
            return p
    input_root = Path('/kaggle/input')
    matches = list(input_root.glob('**/scrfd-kaggle-baseline-py312-cu128.tar.gz'))
    if matches:
        return matches[0]
    raise FileNotFoundError('Could not find packed env archive in /kaggle/input')

repo_source = find_repo_source(REPO_SOURCE)
data_root = find_data_root(DATA_ROOT)
packed_env_archive = find_packed_env(PACKED_ENV_ARCHIVE)

repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
if repo_root.exists():
    shutil.rmtree(repo_root)

if repo_source.is_dir():
    source_root = repo_source / 'scrfd-fullsearch-kaggle'
    shutil.copytree(source_root if source_root.exists() else repo_source, repo_root)
else:
    with zipfile.ZipFile(repo_source) as zf:
        zf.extractall('/kaggle/working')

for shell_dir in ['scripts', 'tools', 'search_tools']:
    for sh_path in (repo_root / shell_dir).glob('*.sh'):
        sh_path.chmod(0o755)

print('Repo source:', repo_source)
print('Data root:', data_root)
print('Packed env archive:', packed_env_archive)


In [ ]:
import subprocess
from pathlib import Path

packed_env_root = Path('/kaggle/working/.scrfd-packed-env')
if packed_env_root.exists():
    shutil.rmtree(packed_env_root)

restore_cmd = [
    'bash',
    str(repo_root / 'scripts' / 'restore_kaggle_packed_env.sh'),
    str(packed_env_archive),
    str(packed_env_root),
]
python_path = subprocess.check_output(restore_cmd, text=True).strip().splitlines()[-1]
print('Restored env python:', python_path)


In [ ]:
import subprocess

work_root = repo_root / 'work_dirs'
result_root = repo_root / 'wouts'
work_root.mkdir(parents=True, exist_ok=True)
result_root.mkdir(parents=True, exist_ok=True)

cmd = [
    python_path,
    str(repo_root / 'scripts' / 'kaggle_competition_entry.py'),
    '--mode', RUN_MODE,
    '--data-root', str(data_root),
    '--work-root', str(work_root),
    '--result-root', str(result_root),
    '--gpu-id', str(GPU_ID),
    '--idx-from', str(IDX_FROM),
    '--idx-to', str(IDX_TO),
    '--template', str(TEMPLATE),
    '--thr', str(THR),
    '--mode-value', str(MODE_VALUE),
    '--total-epochs', str(TOTAL_EPOCHS),
    '--eval-interval', str(EVAL_INTERVAL),
    '--checkpoint-interval', str(CHECKPOINT_INTERVAL),
    '--skip-offline-install',
]
if CHECKPOINT:
    cmd.extend(['--checkpoint', CHECKPOINT])

env = os.environ.copy()
env['PYTHONPATH'] = str(repo_root) if not env.get('PYTHONPATH') else str(repo_root) + ':' + env['PYTHONPATH']

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True, env=env)
